In [1]:
import pandas as pd
import numpy as np
import sqlite3
# Conectamos con la base de datos eventos.db
connection = sqlite3.connect('eventos.db')

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)


In [2]:
# Borramos tabla para pruebas
query = f'drop table if exists eventos'
crsr.execute(query);
# query = f'drop table if exists families'
# crsr.execute(query);
# query = f'drop table if exists family_members'
# crsr.execute(query);
# query = f'drop table if exists user_favourite_events'
# crsr.execute(query);
# query = f'drop table if exists user_selected_recomendations'
# crsr.execute(query);
# query = f'drop table if exists users'
# crsr.execute(query);

In [3]:
# Crear tabla eventos
sql_cre = f'''
CREATE TABLE eventos (
    id INTEGER PRIMARY KEY,

    business_id VARCHAR(50),
    fuente VARCHAR(255),
    external_id VARCHAR(255),

    title VARCHAR(255),
    description TEXT,

    categoria VARCHAR(100),
    tipo_plantilla VARCHAR(100),

    municipio VARCHAR(100),
    territorio VARCHAR(50),
    address VARCHAR(255),

    lat DECIMAL(10,7),
    lng DECIMAL(10,7),

    telefono VARCHAR(50),
    email VARCHAR(255),
    website VARCHAR(255),

    es_interior BOOLEAN,
    es_carrito BOOLEAN,
    es_cambiador BOOLEAN,
    es_silla_ruedas BOOLEAN,
    es_mascotas BOOLEAN,

    edad_minima INT,

    fecha_inicio DATE,
    fecha_fin DATE,

    lugar VARCHAR(255),
    price VARCHAR(50),
    imagen_url TEXT,
    tipo_evento VARCHAR(100)
);
'''

crsr.execute(sql_cre)

In [ ]:
# 1. Leer CSV
df = pd.read_csv("../data/descargas/events_2026-06-02.csv", sep=";")

# 2. Reemplazar NaN por None (para que SQL los acepte como NULL)
df = df.where(pd.notnull(df), None)

# 3. Preparar columnas y placeholders
columnas = [col for col in df.columns if col != "id"]
cols_sql = ", ".join(columnas)
placeholders = ", ".join(["?"] * len(columnas))   # Si usas SQLite
# Para MySQL sería: "%s" en vez de "?"

sql_ins = f"INSERT INTO eventos ({cols_sql}) VALUES ({placeholders})"

# 4. Insertar fila a fila
for _, fila in df.iterrows():
    valores = tuple(fila[columnas])
    crsr.execute(sql_ins, valores)

# crsr.commit()
print("Inserción completada")

Inserción completada


In [9]:
# Comprobamos update
query = '''
select *
    from eventos;
'''
sql_query(query)

,id,business_id,fuente,external_id,title,description,categoria,tipo_plantilla,municipio,territorio,...,es_cambiador,es_silla_ruedas,es_mascotas,edad_minima,fecha_inicio,fecha_fin,lugar,price,imagen_url,tipo_evento
0,1,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,Dendaraba,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,...,1,1,0,0,2026-06-02,None,None,0,None,None
1,2,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,El Boulevard,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,...,1,1,0,0,2026-06-02,None,None,0,None,None
2,3,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,Parque Comercial Gorbeia,None,comercio,Centros comerciales,Zigoitia,araba,...,1,1,0,0,2026-06-02,None,None,0,None,None
3,4,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,Lakua Centro,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,...,1,1,0,0,2026-06-02,None,None,0,None,None
4,5,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,Artea,None,comercio,Centros comerciales,Leioa,bizkaia,...,0,1,0,0,2026-06-02,None,None,0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4065,4066,google_places,parque-kobaundi,None,Kobaundi,None,naturaleza,Parques y jardines,Arrasate / Mondragón,gipuzkoa,...,0,0,1,4,2026-06-02,None,None,0,None,None
4066,4067,google_places,parque-parque-de-el-prado,None,Parque de El Prado,None,naturaleza,Parques y jardines,Vitoria-Gasteiz,araba,...,0,1,1,0,2026-06-02,None,None,0,None,None
4067,4068,google_places,parque-ariznabarra-parkea-parque-de-ariznavarra,None,Ariznabarra Parkea/Parque de Ariznavarra,None,naturaleza,Parques y jardines,Vitoria-Gasteiz,araba,...,0,1,1,0,2026-06-02,None,None,0,None,None
4068,4069,google_places,parque-urkulu-urtegia-embalse-urkulu,None,Urkulu Urtegia/Embalse Urkulu,None,naturaleza,Parques y jardines,Aretxabaleta,gipuzkoa,...,0,1,1,0,2026-06-02,None,None,0,None,None


In [10]:
# Crear tabla families
sql_cre = '''
CREATE TABLE IF NOT EXISTS families (
    id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    family_name VARCHAR(100) NOT NULL
);
'''

crsr.execute(sql_cre)
  


In [11]:
# 1. Leer CSV
df = pd.read_csv("datos/families.csv", sep=";")

# 2. Reemplazar NaN por None (para que SQL los acepte como NULL)
df = df.where(pd.notnull(df), None)

# 3. Preparar columnas y placeholders
columnas = [col for col in df.columns if col != "id"]
cols_sql = ", ".join(columnas)
placeholders = ", ".join(["?"] * len(columnas))   # Si usas SQLite
# Para MySQL sería: "%s" en vez de "?"

sql_ins = f"INSERT INTO families ({cols_sql}) VALUES ({placeholders})"

# 4. Insertar fila a fila
for _, fila in df.iterrows():
    valores = tuple(fila[columnas])
    crsr.execute(sql_ins, valores)

# crsr.commit()
print("Inserción completada")

Inserción completada


In [12]:
# Crear tabla families_members
sql_cre = '''
CREATE TABLE IF NOT EXISTS family_members (
    id INTEGER PRIMARY KEY,
    family_id INTEGER NOT NULL,
    name VARCHAR(100),
    age INTEGER NOT NULL
);
'''

crsr.execute(sql_cre)

In [13]:
# 1. Leer CSV
df = pd.read_csv("datos/family_members.csv", sep=";")

# 2. Reemplazar NaN por None (para que SQL los acepte como NULL)
df = df.where(pd.notnull(df), None)

# 3. Preparar columnas y placeholders
columnas = [col for col in df.columns if col != "id"]
cols_sql = ", ".join(columnas)
placeholders = ", ".join(["?"] * len(columnas))   # Si usas SQLite
# Para MySQL sería: "%s" en vez de "?"

sql_ins = f"INSERT INTO family_members ({cols_sql}) VALUES ({placeholders})"

# 4. Insertar fila a fila
for _, fila in df.iterrows():
    valores = tuple(fila[columnas])
    crsr.execute(sql_ins, valores)

# crsr.commit()
print("Inserción completada")

Inserción completada


In [14]:
# Crear tabla favorite_events
sql_cre = '''
CREATE TABLE IF NOT EXISTS user_favorite_events (
    user_id INTEGER NOT NULL,
    event_id INTEGER NOT NULL,
    saved_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (user_id, event_id)
);
'''

crsr.execute(sql_cre)

In [15]:
# 1. Leer CSV
df = pd.read_csv("datos/user_favorite_events.csv", sep=";")

# 2. Reemplazar NaN por None (para que SQL los acepte como NULL)
df = df.where(pd.notnull(df), None)

# 3. Preparar columnas y placeholders
columnas = [col for col in df.columns if col != "id"]
cols_sql = ", ".join(columnas)
placeholders = ", ".join(["?"] * len(columnas))   # Si usas SQLite
# Para MySQL sería: "%s" en vez de "?"

sql_ins = f"INSERT INTO user_favorite_events ({cols_sql}) VALUES ({placeholders})"

# 4. Insertar fila a fila
for _, fila in df.iterrows():
    valores = tuple(fila[columnas])
    crsr.execute(sql_ins, valores)

# crsr.commit()
print("Inserción completada")

Inserción completada


In [16]:
# Crear tabla selected_recommendations 
sql_cre = '''
CREATE TABLE IF NOT EXISTS user_selected_recommendations (
    user_id INTEGER NOT NULL,
    event_id INTEGER NOT NULL,
    selected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    rating INTEGER,
    PRIMARY KEY (user_id, event_id)
);
'''

crsr.execute(sql_cre)

In [18]:
# 1. Leer CSV
df = pd.read_csv("datos/user_selected_recommendations.csv", sep=";")

# 2. Reemplazar NaN por None (para que SQL los acepte como NULL)
df = df.where(pd.notnull(df), None)

# 3. Preparar columnas y placeholders
columnas = [col for col in df.columns if col != "id"]
cols_sql = ", ".join(columnas)
placeholders = ", ".join(["?"] * len(columnas))   # Si usas SQLite
# Para MySQL sería: "%s" en vez de "?"

sql_ins = f"INSERT INTO user_selected_recommendations ({cols_sql}) VALUES ({placeholders})"

# 4. Insertar fila a fila
for _, fila in df.iterrows():
    valores = tuple(fila[columnas])
    crsr.execute(sql_ins, valores)

# crsr.commit()
print("Inserción completada")

Inserción completada


In [5]:
connection.commit() # Commit por si nos queda algo pendiente antes de cerrar conexión
connection.close()

In [ ]:
import sqlite3

try:
    conn = sqlite3.connect("eventos.db")
    print("OK: la base de datos se abre correctamente")
    conn.close()
except Exception as e:
    print("ERROR:", e)

OK: la base de datos se abre correctamente
